# Phase 4: Full-Dataset Inference

1. Load the trained model + tokenizer, rebuild the exact same architecture, load the saved checkpoint weights.
2. Load the full cleaned dataset (~944k rows) — this is the target for predictions, not the labeled 2,099.
3. Batch inference — run the model over all ~944k reviews in batches (not one at a time), storing predicted class (0/1/2) per aspect per review.
4. Attach predictions back to the dataframe — one column per aspect with the predicted label.
5. Aggregate to product × skin_type level — for each (product_id, skin_type) pair, compute mention counts and % positive/negative per aspect. This is the actual table your app will query.
6. Sanity-check — pick a few well-known products and eyeball whether the aggregated sentiment looks plausible.
7. Save the per-review predictions and the aggregated summary table.

### Step 1: Load model and tokenizer

In [4]:
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertModel
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import random
import numpy as np


### Step 2: Define multihead classifier

In [6]:
# Description: Define multi-head classifier using DistilBERT plus one linear head per aspect
class MultiAspectClassifier(nn.Module):
    def __init__(self, model_name, aspects, num_classes=3, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(model_name)
        self.aspects = aspects
        hidden_size = self.bert.config.hidden_size  # typically 768 for distilbert-base

        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({
            aspect: nn.Linear(hidden_size, num_classes) for aspect in aspects
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # use [CLS] token representation
        pooled = self.dropout(pooled)

        logits = {aspect: self.heads[aspect](pooled) for aspect in self.aspects}
        return logits

In [7]:

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MODEL_NAME = "distilbert-base-uncased"
ASPECTS = ["texture", "scent", "breakouts", "irritation", "efficacy", "packaging", "value"]
MAX_LEN = 256

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

model = MultiAspectClassifier(MODEL_NAME, ASPECTS).to(DEVICE)
checkpoint = torch.load("aspect_sentiment_model_final.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model loaded.")

Model loaded.


### Step 3: Load the full dataset

In [8]:
import pandas as pd

df_full = pd.read_csv("skincare_reviews_clean.csv")
print(df_full.shape)

(944546, 15)


In [9]:


class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

def run_inference(df, text_col="review_text", batch_size=64):
    dataset = InferenceDataset(df[text_col].tolist(), tokenizer, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=batch_size)

    all_preds = {aspect: [] for aspect in ASPECTS}

    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            logits = model(input_ids, attention_mask)

            for aspect in ASPECTS:
                preds = torch.argmax(logits[aspect], dim=1).cpu().numpy()
                all_preds[aspect].extend(preds)

    return all_preds

### Step 4: Run on the small sample

In [10]:
sample = df_full.sample(500, random_state=1).reset_index(drop=True)
preds_sample = run_inference(sample, batch_size=64)

for aspect in ASPECTS:
    sample[f"{aspect}_pred"] = preds_sample[aspect]

sample[["review_text"] + [f"{a}_pred" for a in ASPECTS]].head(10)

100%|██████████| 8/8 [00:06<00:00,  1.14it/s]


,review_text,texture_pred,scent_pred,breakouts_pred,irritation_pred,efficacy_pred,packaging_pred,value_pred
0,It seems like the opposite of what you would e...,2,0,0,0,2,0,0
1,Best facial oil ever. Super moisturizing and a...,1,0,0,0,1,0,1
2,"I love this product! I use it on my neck, and ...",1,0,0,0,0,0,0
3,I’ve been using this product for around a mont...,1,0,0,0,1,0,0
4,Loved how absorbent this was! I used it first ...,1,1,0,0,0,0,0
5,White residue and difficult to remove. A thin ...,2,0,0,0,2,0,0
6,"I’ve been using this for a week, and the secon...",2,0,0,0,1,0,0
7,I received this product through Influenster an...,1,1,0,0,1,1,0
8,Wanted to believe the hype but this balm does ...,2,0,0,0,2,0,0
9,"Very light on the skin, easy to feel and use. ...",1,1,0,0,0,1,0


### Step 5: Run inference on the full dataset

Now that the smoke test works, we run inference on every review in the cleaned dataset.

The model predicts one sentiment label for each of the seven aspects:

- Texture
- Scent
- Breakouts
- Irritation
- Efficacy
- Packaging
- Value

Predictions are processed in batches to avoid running out of memory.

In [11]:
# Run inference on the entire dataset

preds_full = run_inference(df_full, batch_size=64)

print("Inference completed.")
print(f"Processed {len(df_full):,} reviews.")

100%|██████████| 14759/14759 [13:01:41<00:00,  3.18s/it]     


Inference completed.
Processed 944,546 reviews.


### Step 6: Convert numeric predictions back to sentiment labels

The model outputs class IDs:

- `0` → Not mentioned
- `1` → Positive
- `2` → Negative

These IDs are converted back into human-readable labels before saving the results.

In [12]:
# Convert class IDs back to sentiment labels

id2label = {
    0: "Not mentioned",
    1: "Positive",
    2: "Negative"
}

for aspect in ASPECTS:
    df_full[f"{aspect}_pred"] = [
        id2label[p] for p in preds_full[aspect]
    ]

print("Prediction labels added.")

Prediction labels added.


### Step 7: Inspect the prediction results

Before saving the dataset, we perform a few quick checks to make sure the predictions look reasonable.

For each aspect, we inspect:

- the distribution of predicted classes
- a few example reviews with their predicted sentiments

In [13]:
# Check prediction distribution for each aspect

for aspect in ASPECTS:
    print(f"\n{'='*20} {aspect.upper()} {'='*20}")
    print(df_full[f"{aspect}_pred"].value_counts())


==================== TEXTURE ====================
texture_pred
Positive         461577
Not mentioned    415687
Negative          67282
Name: count, dtype: int64

==================== SCENT ====================
scent_pred
Not mentioned    704695
Positive         200593
Negative          39258
Name: count, dtype: int64

==================== BREAKOUTS ====================
breakouts_pred
Not mentioned    793124
Positive         110121
Negative          41301
Name: count, dtype: int64

==================== IRRITATION ====================
irritation_pred
Not mentioned    836212
Positive          79327
Negative          29007
Name: count, dtype: int64

==================== EFFICACY ====================
efficacy_pred
Positive         714149
Not mentioned    148074
Negative          82323
Name: count, dtype: int64

==================== PACKAGING ====================
packaging_pred
Not mentioned    848653
Positive          58170
Negative          37723
Name: count, dtype: int64

===============

In [14]:
# Show a few example predictions

example_columns = [
    "review_text",
    "texture_pred",
    "scent_pred",
    "breakouts_pred",
    "irritation_pred",
    "efficacy_pred",
    "packaging_pred",
    "value_pred"
]

df_full[example_columns].sample(5, random_state=42)

,review_text,texture_pred,scent_pred,breakouts_pred,irritation_pred,efficacy_pred,packaging_pred,value_pred
344257,I have combo skin and this felt incredible on....,Positive,Not mentioned,Not mentioned,Not mentioned,Positive,Not mentioned,Positive
30231,I like this product it is very light weight. y...,Positive,Not mentioned,Not mentioned,Not mentioned,Not mentioned,Not mentioned,Not mentioned
31385,I like this quite well. It’s not something I p...,Not mentioned,Not mentioned,Not mentioned,Positive,Positive,Not mentioned,Not mentioned
383468,The only reason I gave this a second star is b...,Not mentioned,Not mentioned,Negative,Not mentioned,Negative,Not mentioned,Not mentioned
41206,I have been looking for a good daily peel and ...,Not mentioned,Not mentioned,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned


### Step 8: Save the prediction dataset

The seven predicted sentiment columns are added to the original dataframe and saved as a new CSV file.

This file will be used in the next phase for business analysis and visualization.

In [15]:
# Save the dataframe with predictions

output_path = "skincare_reviews_with_predictions.csv"

df_full.to_csv(output_path, index=False)

print(f"Saved predictions to: {output_path}")
print(f"Rows: {len(df_full):,}")

Saved predictions to: skincare_reviews_with_predictions.csv
Rows: 944,546


### Step 9: Verify the saved dataset

Reload the saved file and confirm that:

- the file was written correctly,
- the prediction columns exist,
- the dataset size matches the original.

In [16]:
# Reload the saved file for verification

df_check = pd.read_csv(output_path)

print(df_check.shape)

prediction_columns = [f"{aspect}_pred" for aspect in ASPECTS]
print(df_check[prediction_columns].head())

/var/folders/n9/h4c74fsd6lx9cgyykjw99rk40000gn/T/ipykernel_54638/3966629780.py:3: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(output_path)


(944546, 22)
    texture_pred     scent_pred breakouts_pred irritation_pred  efficacy_pred  \
0       Positive  Not mentioned  Not mentioned   Not mentioned       Positive   
1  Not mentioned  Not mentioned  Not mentioned   Not mentioned       Negative   
2  Not mentioned  Not mentioned  Not mentioned   Not mentioned       Positive   
3       Positive       Positive  Not mentioned   Not mentioned  Not mentioned   
4  Not mentioned  Not mentioned  Not mentioned   Not mentioned       Positive   

  packaging_pred     value_pred  
0  Not mentioned       Positive  
1  Not mentioned  Not mentioned  
2  Not mentioned  Not mentioned  
3  Not mentioned  Not mentioned  
4  Not mentioned       Positive  


In [17]:
# --- Step 10: Aggregate predictions to product x skin_type level ---
import pandas as pd

df_full = pd.read_csv("skincare_reviews_with_predictions.csv", dtype={"author_id": str}, low_memory=False)

ASPECTS = ["texture", "scent", "breakouts", "irritation", "efficacy", "packaging", "value"]

def aggregate_aspect(df, aspect, group_cols=["product_id", "product_name", "skin_type"]):
    col = f"{aspect}_pred"
    counts = df.groupby(group_cols)[col].value_counts().unstack(fill_value=0)

    # Ensure all three categories exist even if a group has zero of one
    for cat in ["Positive", "Negative", "Not mentioned"]:
        if cat not in counts.columns:
            counts[cat] = 0

    counts["total_mentions"] = counts["Positive"] + counts["Negative"]
    counts["pct_positive"] = (counts["Positive"] / counts["total_mentions"]).where(counts["total_mentions"] > 0)
    counts["pct_negative"] = (counts["Negative"] / counts["total_mentions"]).where(counts["total_mentions"] > 0)

    counts = counts.rename(columns={
        "Positive": f"{aspect}_positive_count",
        "Negative": f"{aspect}_negative_count",
        "Not mentioned": f"{aspect}_not_mentioned_count",
        "total_mentions": f"{aspect}_total_mentions",
        "pct_positive": f"{aspect}_pct_positive",
        "pct_negative": f"{aspect}_pct_negative",
    })
    return counts.reset_index()

# Build the summary table by merging each aspect's aggregation together
summary = None
for aspect in ASPECTS:
    agg = aggregate_aspect(df_full, aspect)
    if summary is None:
        summary = agg
    else:
        summary = summary.merge(agg, on=["product_id", "product_name", "skin_type"], how="outer")

print(summary.shape)
summary.head()

(9004, 45)


,product_id,product_name,skin_type,texture_negative_count,texture_not_mentioned_count,texture_positive_count,texture_total_mentions,texture_pct_positive,texture_pct_negative,scent_negative_count,...,packaging_positive_count,packaging_total_mentions,packaging_pct_positive,packaging_pct_negative,value_negative_count,value_not_mentioned_count,value_positive_count,value_total_mentions,value_pct_positive,value_pct_negative
0,P107306,Renewing Eye Cream,Combination,3,79,29,32,0.906250,0.093750,1,...,3,10,0.300000,0.700000,22,62,27,49,0.551020,0.448980
1,P107306,Renewing Eye Cream,Dry,2,28,10,12,0.833333,0.166667,0,...,1,3,0.333333,0.666667,5,24,11,16,0.687500,0.312500
2,P107306,Renewing Eye Cream,Normal,0,22,5,5,1.000000,0.000000,0,...,0,3,0.000000,1.000000,1,22,4,5,0.800000,0.200000
3,P107306,Renewing Eye Cream,Oily,0,18,4,4,1.000000,0.000000,0,...,0,2,0.000000,1.000000,6,13,3,9,0.333333,0.666667
4,P107306,Renewing Eye Cream,Unknown,4,41,8,12,0.666667,0.333333,0,...,0,1,0.000000,1.000000,9,35,9,18,0.500000,0.500000


In [18]:
# Save the summary table for the app
summary.to_csv("aspect_sentiment_summary.csv", index=False)
print("Saved aspect_sentiment_summary.csv:", summary.shape)

Saved aspect_sentiment_summary.csv: (9004, 45)


In [19]:
top_product = df_full["product_name"].value_counts().index[0]
print(top_product)
summary[summary["product_name"] == top_product]

Lip Sleeping Mask Intense Hydration with Vitamin C


,product_id,product_name,skin_type,texture_negative_count,texture_not_mentioned_count,texture_positive_count,texture_total_mentions,texture_pct_positive,texture_pct_negative,scent_negative_count,...,packaging_positive_count,packaging_total_mentions,packaging_pct_positive,packaging_pct_negative,value_negative_count,value_not_mentioned_count,value_positive_count,value_total_mentions,value_pct_positive,value_pct_negative
1711,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,Combination,542,5374,2250,2792,0.805874,0.194126,299,...,617,841,0.733650,0.266350,807,4962,2397,3204,0.748127,0.251873
1712,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,Dry,194,2037,813,1007,0.807349,0.192651,109,...,233,313,0.744409,0.255591,345,1815,884,1229,0.719284,0.280716
1713,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,Normal,141,1452,612,753,0.812749,0.187251,86,...,166,230,0.721739,0.278261,237,1325,643,880,0.730682,0.269318
1714,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,Oily,132,1255,542,674,0.804154,0.195846,83,...,141,194,0.726804,0.273196,226,1111,592,818,0.723716,0.276284
1715,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,Unknown,29,508,178,207,0.859903,0.140097,17,...,59,76,0.776316,0.223684,76,442,197,273,0.721612,0.278388


## Phase 4 Summary

The trained multi-head DistilBERT model was used to predict sentiment for seven skincare aspects across the full review dataset. Predictions were saved per review and aggregated by product and skin type for downstream analysis.

### Completed steps
- Loaded the trained model checkpoint.
- Loaded the cleaned review dataset.
- Built an inference pipeline using batched processing.
- Predicted sentiments for all seven skincare aspects.
- Converted numeric predictions back into readable labels.
- Added seven prediction columns to the original dataset.
- Saved the enriched dataset for downstream analysis.

The resulting dataset now contains aspect-level sentiment predictions for every review and is ready for exploratory analysis, visualization, and business insights in the next phase of the project.

### Limitations
- Packaging negatives are likely over-predicted on the full dataset (39% of mentioned-packaging
  reviews are predicted negative vs. ~25% in the LLM-labeled random sample), because the
  training set was enriched with negative packaging reviews. Treat packaging percentages as
  approximate.
- Predictions are model-generated estimates, not verified labels for every review.
- The model was trained on a relatively small labeled sample, so performance may vary across products, writing styles, and aspects.
- Packaging predictions may be biased toward negative sentiment because negative packaging examples were overrepresented in the training data. Treat those percentages as approximate.
- Aspect percentages are calculated among reviews where that aspect was predicted as mentioned; they do not represent all reviews for a product.
- Reviews are truncated to 256 tokens during inference, so sentiment expressed later in a long review may be missed.
- Aggregated results depend on the available product and skin-type metadata and should be interpreted with those data limitations in mind.

